# RAE JAX CelebA Kaggle TPU v5e-8 Pipeline (SiTDH-B Resume)

Notebook này là bản resume-only cho trường hợp bạn giải nén lại output của notebook cũ vào `/kaggle/working`, nên chỉ còn phải dựng lại môi trường `uv` trong session Kaggle mới.

Artifact cần có sẵn sau bước giải nén output archive:

- `/kaggle/working/RAE`
- `/kaggle/working/celeba256_imgfolder`
- `/kaggle/working/celeba256_val_fid_stats_cpu.pkl`
- `/kaggle/working/celeba256_source_gmm_pyr16k.npz`
- `/kaggle/working/results_jax_tpu/` với ít nhất một thư mục run khớp mẫu `CelebA256_SiTDH-B_DINOv2-B_moe1_jax_tpuv5e8-*` và bên trong có `checkpoint_*`

Notebook này bắt đầu bằng cell giải nén `_output_.zip` của notebook cũ, sau đó mới kiểm tra repo, chạy `uv sync`, lấy Kaggle secret cho wandb, tự tìm run Orbax mới nhất và checkpoint mới nhất bên trong nó, copy checkpoint đó sang một workdir resume mới có timestamp, seed một `wandb_run.json` mới với run id mới, rồi resume train từ workdir mới nhưng vẫn giữ nguyên `exp_name` gốc của run đã tạo checkpoint.

Flow này tách riêng việc restore checkpoint khỏi việc tái dùng W&B run cũ: mỗi lần resume sẽ có W&B run mới sạch lịch sử, nhưng tên run vẫn khớp run đã tạo checkpoint để dễ đối chiếu lineage. Runtime vẫn xóa ngay `checkpoint_*` vừa restore trong workdir mới sau khi state đã load vào RAM, nên không cần giữ đồng thời checkpoint cũ và checkpoint mới trong cùng workdir resume.


In [ ]:
!unzip -o /kaggle/input/notebooks/kieuhongquan/rae-jax/_output_.zip -d /kaggle/working 1>out.txt 2>err.txt


In [ ]:
%%bash
set -euo pipefail

[ -d /kaggle/working/RAE/.git ]
cd /kaggle/working/RAE
git pull
git rev-parse --short HEAD


In [ ]:
import os

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

%cd /kaggle/working/RAE
!uv sync -q


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE
uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["WANDB_KEY"] = user_secrets.get_secret("WANDB_KEY")


In [ ]:
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
celeba_root = Path("/kaggle/working/celeba256_imgfolder")
fid_stats_path = Path("/kaggle/working/celeba256_val_fid_stats_cpu.pkl")
source_gmm_path = Path("/kaggle/working/celeba256_source_gmm_pyr16k.npz")
results_root = Path("/kaggle/working/results_jax_tpu")
run_dirs = sorted(results_root.glob("CelebA256_SiTDH-B_DINOv2-B_moe1_jax_tpuv5e8-*"))
source_workdir = run_dirs[-1] if run_dirs else None
checkpoint_dirs = sorted(source_workdir.glob("checkpoint_*"), key=lambda path: int(path.name.split("_")[-1])) if source_workdir else []
latest_ckpt_dir = checkpoint_dirs[-1] if checkpoint_dirs else None

for path in [repo_root, celeba_root, fid_stats_path, source_gmm_path, results_root]:
    print(path, "exists=", path.exists())
print("source_workdir=", source_workdir)
print("latest_ckpt_dir=", latest_ckpt_dir)

assert repo_root.exists()
assert celeba_root.exists()
assert fid_stats_path.exists()
assert source_gmm_path.exists()
assert results_root.exists()
assert source_workdir is not None
assert latest_ckpt_dir is not None


## Resume Stage 2

Cell bên dưới tự tìm thư mục run mới nhất trong `/kaggle/working/results_jax_tpu`, copy thư mục `checkpoint_*` mới nhất sang một workdir resume mới có timestamp, tạo `wandb_run.json` mới với W&B run id mới, rồi resume train từ workdir mới đó. `exp_name` vẫn giữ nguyên theo run gốc đã tạo checkpoint để lineage dễ đọc, còn history W&B của lần resume sẽ sạch và tách hẳn khỏi run cũ.


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

results_root="/kaggle/working/results_jax_tpu"
source_workdir=$(find "${results_root}" -maxdepth 1 -mindepth 1 -type d -name "CelebA256_SiTDH-B_DINOv2-B_moe1_jax_tpuv5e8-*" | sort -V | tail -n 1)
[ -n "${source_workdir}" ]
latest_ckpt=$(find "${source_workdir}" -maxdepth 1 -type d -name "checkpoint_*" | sort -V | tail -n 1)
[ -n "${latest_ckpt}" ]

timestamp="$(TZ=Asia/Bangkok date +%Y%m%d-%H%M%S)"
resume_exp_name=$(basename "${source_workdir}")
resume_workdir="${results_root}/${resume_exp_name}-resume-${timestamp}"

mkdir -p "${resume_workdir}"
cp -a "${latest_ckpt}" "${resume_workdir}/"

new_run_id="$(python3 - <<'PY'
import os
print(os.urandom(4).hex())
PY
)"

cat > "${resume_workdir}/wandb_run.json" <<EOF
{"run_id":"${new_run_id}","entity":"TungBangDSLab","project":"moe-diffusion","exp_name":"${resume_exp_name}"}
EOF

echo "source workdir: ${source_workdir}"
echo "copied checkpoint: ${latest_ckpt}"
echo "new resume workdir: ${resume_workdir}"
echo "new exp_name: ${resume_exp_name}"
echo "new wandb run_id: ${new_run_id}"

export ENTITY="TungBangDSLab"
export PROJECT="moe-diffusion"
export RAE_JAX_REBUILD_BACKEND=1

uv run python src_jax/train.py \
  --config configs/stage2/training/CelebA256_SiTDH-B_DINOv2-B_moe1_jax_tpuv5e8.yaml \
  --data-path /kaggle/working/celeba256_imgfolder \
  --workdir "${resume_workdir}" \
  --exp-name "${resume_exp_name}" \
  --precision bf16 \
  --wandb \
  --wandb-entity TungBangDSLab \
  --wandb-project "${PROJECT}" \
  --set training.global_batch_size=64 \
  --set training.num_workers=16 \
  --set training.prefetch_factor=4 \
  --set training.ckpt_every=170000 \
  --set training.log_rae_latent_stats=true \
  --set training.log_activation_stats=true \
  --set eval.prefetch_factor=4 \
  --set eval.fid_ref=/kaggle/working/celeba256_val_fid_stats_cpu.pkl \
  --set eval.fid_every=10000 \
  --set eval.fid_num_samples=4096
